# Project Overview
This project focuses on analyzing health insurance claims data and building a machine learning model to predict claim outcomes (e.g., approved, denied, pending). The goal is to identify key factors influencing claim status and evaluate model performance using classification techniques.


# Overview
Perform Exploratory Data Analysis (EDA) on insurance claims data

Clean and preprocess the dataset
Engineer useful features to improve prediction
Build classification models to predict claim status
Evaluate model performance using accuracy and classification metrics

# Objective
Perform Exploratory Data Analysis (EDA) on insurance claims data
Clean and preprocess the dataset
Engineer useful features to improve prediction
Build classification models to predict claim status
Evaluate model performance using accuracy and classification metrics

# Dataset
The dataset contains health insurance claim information including: Dataset used - https://www.kaggle.com/datasets/leandrenash/enhanced-health-insurance-claims-dataset/data

Claim amount
Patient age
Patient income
Claim type
Provider location
Claim submission method
Claim status (target variable)

## Loading Data

In [ ]:
#Data Loaded

#Load & Inspect Data
# ===============================
# STEP 0: Import Libraries
# ===============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, cross_val_score
# ===============================
# STEP 1: Load Data
# ===============================
df = pd.read_csv('enhanced_health_insurance_claims.csv')
                
print("First 5 rows:")
print(df.head())

# ===============================
# STEP 2: Basic Info
# ===============================
print("\nDataset Info:")
df.info()

print("\nSummary Statistics:")
df.describe().T


## Data Cleaning

Checks missing values Removes duplicates
Fills numeric missing values with the median Fills 
categorical missing values with the mode Verifies the result

In [ ]:
# ===============================
# DATA CLEANING
# ===============================

print("Missing values BEFORE cleaning:")
print(df.isnull().sum().sort_values(ascending=False))

# Drop duplicates
print("\nDuplicates before:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicates after:", df.duplicated().sum())

# -------------------------------
# Drop ID columns
# -------------------------------
id_cols = ['ClaimID', 'PatientID', 'ProviderID']
df = df.drop(
    columns=[col for col in id_cols if col in df.columns],
    errors='ignore'
)

# -------------------------------
# Fill numeric columns with median
# -------------------------------
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# -------------------------------
# Fill categorical columns with mode
# -------------------------------
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# -------------------------------
# Final check
# -------------------------------
print("\nMissing values AFTER cleaning:")
print(df.isnull().sum().sort_values(ascending=False))

## Claim Status Distribution Analysis 
This section analyzes the distribution of the target variable 
ClaimStatus to identify class balance and understand the
frequency of each claim category.

### Frequency Distribution

In [ ]:

# ===============================
# TARGET DISTRIBUTION ANALYSIS
# ===============================

target_column = "ClaimStatus"

counts = df[target_column].value_counts()
percentages = (counts / counts.sum()) * 100

print("Value counts:", counts)

print("\nPercentage distribution:")
print(percentages)

# Prepare DataFrame for seaborn
plot_df = counts.reset_index()
plot_df.columns = [target_column, "Count"]

# Plot
plt.figure(figsize=(8,5))
sns.barplot(data=plot_df, x=target_column, y="Count")

# Add percentage labels 
for i, v in enumerate(counts.values):
    plt.text(i, v, f"{percentages.iloc[i]:.1f}%", 
             ha='center', va='bottom')

plt.title("Claim Status Distribution")
plt.xlabel("Claim Status")
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## Another Categorical Variable

In [ ]:
col = "ClaimType"

counts = df[col].value_counts()

plt.figure(figsize=(8,5))

sns.barplot(
    x=counts.index,
    y=counts.values
)

plt.title("Claim Type Distribution")
plt.xlabel("Claim Type")
plt.ylabel("Count")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

print(df.select_dtypes(include='object').columns)

### Percentage Distribution Visualization

In [ ]:
plt.figure(figsize=(8,5))

# Plot bars
plt.bar(counts.index.astype(str), counts.values)

# Add percentage labels
for i, v in enumerate(counts.values):
    plt.text(i, v + max(counts.values)*0.01,
             f"{(v / counts.sum())*100:.1f}%",
             ha='center')

plt.title("Claim Status Distribution")
plt.xlabel("Claim Status")
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

## Numerical Feature Distribution Analysis

This section visualizes the distribution of numerical variables in the dataset using histograms to better understand data spread, skewness, and potential outliers.

    - plotting histograms
    - examining distributions
    - checking spread/skewness/outliers.

In [ ]:
#EDA -Numerical Features Distribution
num_cols = df.select_dtypes(include=np.number).columns

df[num_cols].hist(figsize=(12,10), bins=20)
plt.suptitle("Numerical Feature Distributions")
plt.tight_layout()
plt.show()

## Correlation Heatmap of Numerical Features
This section visualizes correlations between numerical features
to identify relationships, dependencies, 
and multicollinearity in the dataset.

In [ ]:
plt.figure(figsize=(10,6))

sns.heatmap(df[num_cols].corr(),
            annot=True,
            cmap="coolwarm",
            fmt=".2f",
            linewidths=0.5)

plt.title("Correlation Heatmap")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

## Outlier Detection Using Box Plots

In [ ]:
col = "ClaimType"
counts = df[col].value_counts().head(10)

plt.figure(figsize=(8,4))

sns.barplot(x=counts.index, y=counts.values)

total = counts.sum()
for i, v in enumerate(counts.values):
    plt.text(i, v, f"{(v/total)*100:.1f}%", ha='center', va='bottom')

plt.title(f"{col} Distribution (Top 10)")
plt.xlabel(col)
plt.ylabel("Count")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


## Feature Engineering & Encoding
- dropping ID columns
- creating new features
- label encoding target
- one-hot encoding

In [ ]:
# ===============================
# Create a new feature that normalizes claim amount by patient age
# ===============================
if 'ClaimAmount' in df.columns and 'PatientAge' in df.columns:
    df['ClaimPerAge'] = df['ClaimAmount'] / (df['PatientAge'] + 1)

# ===============================
# ENCODE TARGET
# ===============================


target_column = "ClaimStatus"
df[target_column] = LabelEncoder().fit_transform(df[target_column].astype(str))

# ===============================
# ONE-HOT ENCODING (SAFE NOW)
# ===============================
df = pd.get_dummies(df, drop_first=True)

## Logistic Regression Model Training and Performance Evaluation

- preprocessing
- feature engineering
- train-test split
- scaling
- logistic regression
- evaluation metrics
- confusion matrix

In [ ]:


# ===============================
# COPY DATA (SAFE PRACTICE)
# ===============================
df = df.copy()

# ===============================
# DEFINE TARGET
# ===============================
target_column = "ClaimStatus"

# ===============================
# SPLIT FEATURES & TARGET
# ===============================
X = df.drop(columns=[target_column])
y = df[target_column]

# ===============================
# ENCODE TARGET
# ===============================
le_target = LabelEncoder()
y = le_target.fit_transform(y)

# ===============================
# ONE-HOT ENCODE FEATURES (IMPORTANT)
# ===============================
X = pd.get_dummies(X, drop_first=True)

# ===============================
# TRAIN-TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ===============================
# FEATURE SCALING
# ===============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ===============================
# MODEL TRAINING
# ===============================
model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

# ===============================
# PREDICTIONS
# ===============================
y_pred = model.predict(X_test)

# ===============================
# EVALUATION
# ===============================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# ===============================
# CONFUSION MATRIX
# ===============================
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix")
plt.show()


## Baseline Model Evaluation Using Dummy Classifier

In [ ]:


# ===============================
# BASELINE MODEL (DUMMY CLASSIFIER)
# ===============================
dummy = DummyClassifier(strategy="most_frequent", random_state=42)

dummy.fit(X_train, y_train)

y_dummy_pred = dummy.predict(X_test)

print("===== BASELINE MODEL (DUMMY CLASSIFIER) =====")
print("Accuracy:", accuracy_score(y_test, y_dummy_pred))
print("\nClassification Report:\n")
#print(classification_report(y_test, y_dummy_pred))
print(classification_report(y_test, y_dummy_pred, zero_division=0))
# Confusion Matrix
ConfusionMatrixDisplay.from_estimator(dummy, X_test, y_test)
plt.title("Confusion Matrix - Dummy Classifier (Baseline)")
plt.show()

 ## Random Forest Model Training and Evaluation
- trains a RandomForest model
- makes predictions
- evaluates accuracy and classification report

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## Random Forest Capstone Model
- full preprocessing pipeline
- feature engineering
- train/test split
- Random Forest model
- evaluation metrics
- confusion matrix
 

In [ ]:
# ===============================
# RANDOM FOREST - CAPSTONE MODEL 
# ===============================

# ===============================
# Fill missing numeric values
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill missing categorical values
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# ===============================
# DEFINE TARGET
# ===============================
target_column = "ClaimStatus"

# ===============================
# SPLIT FEATURES AND TARGET
# ===============================
X = df.drop(columns=[target_column])
y = df[target_column]

# ===============================
# ENCODE TARGET
# ===============================
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y)

# ===============================
# ONE-HOT ENCODE FEATURES (BEST PRACTICE)
# ===============================
X = pd.get_dummies(X, drop_first=True)

# ===============================
# TRAIN-TEST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Save feature names (important for interpretation)
feature_names = X.columns

# ===============================
# MODEL TRAINING
# ===============================
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

# ===============================
# PREDICTIONS
# ===============================
y_pred = model.predict(X_test)

# ===============================
# EVALUATION
# ===============================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# ===============================
# CONFUSION MATRIX
# ===============================
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix - Random Forest")
plt.show()



## Correlation with Claim Status
- encodes target
- calculates correlation
- ranks features based on relationship with target

In [ ]:

# ===============================
# COPY DATA SAFELY
# ===============================
df_corr = df.copy()

# ===============================
# ENCODE TARGET VARIABLE
# ===============================
le_corr = LabelEncoder()
df_corr["ClaimStatus"] = le_corr.fit_transform(df_corr["ClaimStatus"].astype(str))

# ===============================
# KEEP ONLY NUMERIC DATA
# ===============================
numeric_df = df_corr.select_dtypes(include='number')

# ===============================
# CORRELATION WITH TARGET
# ===============================
corr_series = numeric_df.corr()["ClaimStatus"]

# Remove self-correlation
corr_series = corr_series.drop("ClaimStatus")

# Sort values
corr_series = corr_series.sort_values(ascending=False)

# ===============================
# PRINT RESULTS
# ===============================
print("Feature correlation with ClaimStatus:")
print(corr_series)

# ===============================
# VISUALIZATION (OPTIONAL BUT RECOMMENDED)
# ===============================
plt.figure(figsize=(10,6))
sns.barplot(x=corr_series.values, y=corr_series.index)

plt.title("Feature Correlation with ClaimStatus")
plt.xlabel("Correlation Value")
plt.ylabel("Features")

plt.tight_layout()
plt.show()


## Confusion Matrix - Random Forest
- plots confusion matrix
- uses normalization (normalize='true')
- evaluates classification performance

In [ ]:


# Confusion matrix
ConfusionMatrixDisplay.from_estimator(
   model,
    X_test,
    y_test,
    normalize='true'
)

plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()
plt.show()

## Cross Validation

The Random Forest model achieved an average cross-validation accuracy of XX%. The low standard deviation indicates that the model performs consistently across multiple subsets of the data and is likely to generalize well to unseen data.

In [ ]:
cv_scores = cross_val_score(
    RandomForestClassifier(random_state=42),
    X,
    y,
    cv=5,
    scoring='accuracy'
)

print("Cross Validation Scores:", cv_scores)
print("Mean CV Accuracy:", cv_scores.mean())
print("Standard Deviation:", cv_scores.std())

# Grid Search Hyperparameter Tuning
Grid Search identified the optimal combination of hyperparameters for the Random Forest model. The tuned model achieved a higher validation score than the default model, indicating improved predictive performance.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy'
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

# Optional: evaluate best model
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("Tuned Model Accuracy:",
      accuracy_score(y_test, y_pred_best))

# ===============================
# FEATURE CORRELATION ANALYSIS
# ===============================
df_corr = df.copy()

## Random Forest Feature Importance Ranking
- extracts feature importance
- selects top 10 features
- visualizes contribution to model

In [ ]:

# ===============================
# FEATURE IMPORTANCE (RANDOM FOREST)
# ===============================

# Get feature importance from trained model
importances = best_model.feature_importances_

# Use saved feature names from training step
features = feature_names

# Create DataFrame
feat_df = pd.DataFrame({
    "Feature": features,
    "Importance": importances
}).sort_values(by="Importance", ascending=True)

# ===============================
# OPTIONAL: KEEP TOP FEATURES ONLY (recommended)
# ===============================
top_n = 10
feat_df_top = feat_df.tail(top_n)

# ===============================
# PLOT FEATURE IMPORTANCE
# ===============================
plt.figure(figsize=(10,6))

plt.barh(feat_df_top["Feature"], feat_df_top["Importance"])

plt.title(f"Top {top_n} Random Forest Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Features")

plt.tight_layout()
plt.show()

## Model Evaluation Using Confusion Matrix
- evaluates predictions
- visualizes confusion matrix
- uses Random Forest model

In [ ]:

# ===============================
# CONFUSION MATRIX
# ===============================
ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test
)

plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.tight_layout()
plt.show()

## Findings

- Claim statuses are relatively balanced across Approved, Denied, and Pending categories.
- Claim Type shows variation in frequency across submitted claims.
- Random Forest outperformed Logistic Regression in predictive accuracy.
- Cross-validation results indicate stable model performance.
- Feature importance analysis identified the variables that most strongly influence claim outcomes.

## Recommendations
- Focus claim review resources on claims with characteristics associated with denials.
- Collect additional predictive features to improve model accuracy.
- Deploy the Random Forest model as a decision-support tool rather than an automated decision system.
- Continue monitoring model performance as new claims data becomes available.


## Next Steps

Future improvements may include:

- Additional feature engineering
- Testing XGBoost
- Handling class imbalance more aggressively
- Collecting more claim history data
- Using probability thresholds rather than hard classifications